# Aceleração com `torch.compile`

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

`torch.compile` traça seu modelo em um grafo FX, otimiza (fusão de operadores, escolha de kernel) e compila para Triton/AOT. Costuma dar 1.3–2× de speed-up grátis em GPUs modernas, sem alterar código.


## Formulação Matemática

Conceitualmente $f_{\text{compiled}} \equiv f$, mas avaliado por um grafo fundido em vez do dispatch eager. A primeira chamada paga o custo de compilação; as seguintes reutilizam o grafo cacheado.


## Implementação


In [ ]:
import time, torch
import torch.nn as nn


In [ ]:
def benchmark(model, x, warmup=5, runs=20):
    if x.is_cuda: torch.cuda.synchronize()
    for _ in range(warmup): _ = model(x)
    if x.is_cuda: torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs): _ = model(x)
    if x.is_cuda: torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mlp = nn.Sequential(nn.Linear(1024, 1024), nn.GELU(), nn.Linear(1024, 1024)).to(device).eval()
x = torch.randn(64, 1024, device=device)


## Experimento


In [ ]:
eager_t = benchmark(mlp, x)
mlp_c = torch.compile(mlp)
compiled_t = benchmark(mlp_c, x)
print(f'eager     : {eager_t*1e3:.2f} ms')
print(f'compiled  : {compiled_t*1e3:.2f} ms')
print(f'speed-up  : {eager_t / max(compiled_t, 1e-9):.2f}x')


## Discussão

- A primeira chamada depois do `torch.compile` leva segundos (trace + compile). Planeje no warmup.
- Use `mode='reduce-overhead'` para inferência pesada, `'max-autotune'` quando puder pagar compile longo.
- Cuidado com graph breaks (branches em Python, operações in-place com side effects) — eles matam o ganho.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
